# 19. Connecting MCP with Agents

**MCP** (Model Context Protocol) is a **standard way to share tools** with an agent.

Instead of writing tool functions inside your own code, MCP lets you connect to a
small **tool server**. The server offers tools (like "get price", "search", "read a file"),
and your agent can use them.

Think of MCP as a **USB port for tools**: any MCP server can plug into any MCP-ready agent.

## Real-life analogy

Imagine a **power socket** on the wall (the MCP server) and a **plug** on your device (the agent).

- The **socket** already provides power (tools).
- Any device with the right **plug** can use it.

You do not rebuild the electricity every time — you just **plug in**.
MCP works the same way: build a tool server once, and any agent can plug in and use it.

## How it works

| Step | What happens |
|------|--------------|
| 1 | We run a small **MCP server** that offers one or more tools |
| 2 | The agent **connects** to that server |
| 3 | AutoGen turns the server's tools into normal agent tools |
| 4 | The agent **calls a tool** when needed and answers you |

The agent's **brain is still OpenAI `gpt-4o-mini`**. MCP only provides the *tools*.

## Step 1 — Make a tiny MCP server

An MCP server is just a small Python program that offers tools.
We write one below using **FastMCP** and save it as a file called `my_mcp_server.py`.

It offers one tool: `get_price(item)` — it returns today's price of a grocery item.
(The LLM cannot guess these made-up prices, so it *must* use the tool.)

In [ ]:
# Write a small MCP server to a file called my_mcp_server.py
server_code = '''
from mcp.server.fastmcp import FastMCP

# Give the server a name
mcp = FastMCP("Grocery")

@mcp.tool()
def get_price(item: str) -> str:
    """Return today's price for a grocery item."""
    prices = {"apple": "30 rupees", "banana": "10 rupees", "milk": "50 rupees"}
    return prices.get(item.lower(), "price not available")

if __name__ == "__main__":
    # Talk over stdio (standard input/output) - the simplest MCP connection
    mcp.run()
'''

with open("my_mcp_server.py", "w") as f:
    f.write(server_code)

print("Saved my_mcp_server.py")

## Step 2 — One-time setup for Jupyter (Windows)

MCP starts the tool server as a **small background program**. Inside Jupyter on Windows,
that background program needs a normal file to write its logs to.

The one line below gives it one (a file called `mcp_server.log`). Just run it once.
You do **not** need to understand this line to learn MCP — it is only a Jupyter detail.

In [ ]:
# One-time setup so MCP can start its background server inside Jupyter.
# It simply points the server's logs at a normal file (mcp_server.log).
import mcp.client.stdio
mcp.client.stdio.stdio_client.__wrapped__.__defaults__ = (open("mcp_server.log", "w"),)
print("MCP setup done")

## Step 2 — Connect the agent to the MCP server

- `StdioServerParams` tells AutoGen **how to start** our server (`python my_mcp_server.py`).
- `mcp_server_tools(...)` **connects** and collects the server's tools.
- We hand those tools to the agent, exactly like normal tools.

In [ ]:
import sys
from dotenv import load_dotenv
load_dotenv()

from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.mcp import StdioServerParams, mcp_server_tools

# The OpenAI brain (same as every other notebook)
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

# 1) How to start our MCP server.
#    We use sys.executable (the SAME Python running this notebook) so the
#    server starts with the same packages installed. Do NOT just write "python"
#    - that can pick a different Python that does not have the mcp library.
#    read_timeout_seconds is raised to 30 so a slow first start does not time out.
server = StdioServerParams(
    command=sys.executable,
    args=["my_mcp_server.py"],
    read_timeout_seconds=30,
)

# 2) Connect and collect the tools it offers
tools = await mcp_server_tools(server)
print("Tools from MCP server:", [t.name for t in tools])

# 3) Give the MCP tools to the agent
shopper = AssistantAgent(
    name="shopper",
    model_client=model_client,
    tools=tools,                    # <-- tools that came from the MCP server
    reflect_on_tool_use=True,       # turn the tool result into a nice sentence
    system_message="You help with grocery prices. Use your tools to check prices.",
)

# 4) Ask something that needs the tool
result = await shopper.run(task="What is the price of milk today?")
print(result.messages[-1].content)

## Key points to remember

- **MCP** = a standard way to **share tools** with agents (like a "USB port for tools").
- An **MCP server** is a small program that offers tools; write one easily with **FastMCP**.
- Connect with **`StdioServerParams`** + **`mcp_server_tools(...)`**, then pass the tools to the agent.
- The agent's **brain stays OpenAI `gpt-4o-mini`** — MCP only adds tools.
- Best part: build a tool server **once**, and **any** MCP-ready agent can plug in and use it.